# De Bruijn graphs

You will be implementing one of the primary assembly algorithms from short-read data that is used today. We will implement a simple form of the algorithm where we _assume perfect sequencing_. That is, everything is sequenced exactly once and there are no errors or variants in the sequencing. 

A graph is composed of **nodes** and **edges** and we will need to develop a data strcture to track edges between nodes in our graph. We have provided the basic class structure as well as descriptions of functions to `add_edge` and `remove_edge` from the graph. You will need to implement these functions in order to then build the de Bruijn graph. 

In our implementation below, we use a `defaultdict` data structure to hold a list of all edges in the graph where all "right" nodes connected to a "left" node are stored in a list for that node.

```
build_debruijn_graph:
define substring length k and input string
For each k-length substring of input:
  split k mer into left and right k-1 mer
  add k-1 mers as nodes with a directed edge from left k-1 mer to right k-1 mer
```

---
## Eulerian walk

To continue our implementation from last class, we will use our De Bruijn graph to output a valid sequence from the assembly. This is implemented as a recursive algorithm by considering all valid edges. You will notice that as you change $k$, we are able to better recapitulate our sequence depending on how repetitive it is. In a more complex implementation of a Eulerian walk there are heuristics and defined rules for determining the validity of traversing a specific edge in the graph to result in a full graph-traversal. One of these methods is to traverse the graph in a depth first manner to avoid sectioning off any part of the graph in the traversal. In our implementation we will ignore these for simplicity.

```
eulerian_walk:
Beginning at first_node as node

For node:
    follow a random valid edge from node
    remove edge
    recurse
```


In [1]:
# Import the tool that can read squished (.gz) files
import gzip
# Open the squished file as text
with gzip.open("data/sacCer_R64_150SE_30x_wgsim.fq.gz", "rt") as f:
    # Repeat 8 times, which is 2 reads at 4 lines each
    for _ in range(8):
        # Read one line and print it without the extra newline
        print(f.readline().rstrip())

@read_1
ATCACAAAGGAGACAAAATCGTTGATAAAAAGTGCATCAACATTCCCAGCCAGAAAATGCACATCATAAAGACATGTTATTCAAGAGCCACGACCGTCTTCAATTTATCTTTTATAAAAAACCCTTGTTCTACTGACAGGATGGAATAGA
+
IIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII
@read_2
TAGATAATTTCAATGCGACTGATTTATTACAAAATCAAGGCCTTGTTATGGAAGAGAAAGAGGATGCTGTAAAGAATGAAGTTCTTCTTAATCGATTTTATCCACTTCAAAATCAGACTTATGACATTGCAGATAAGTCTGGCCTTAAAG
+
IIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII


In [2]:
import gzip

def read_fastq(filename):
    """Read sequences from a FASTQ file.
    Args:
        filename (str): Path to FASTQ file.
    Returns:
        list: List of DNA sequence strings (quality scores ignored).
    """
    # Open the file: gzip.open for .gz (compressed) files and plain text
    opener = gzip.open if filename.endswith(".gz") else open
    # Empty list to hold the DNA reads
    sequences = []
    # Open the file in text mode ("rt") and close it automatically when done
    with opener(filename, "rt") as fastq:
        # Go through the file one line at a time, counting lines starting at 1
        for line_count, line in enumerate(fastq, 1):
            # FASTQ has 4 lines per read; the DNA is line 2, 6, 10, ... (remainder 2)
            if line_count % 4 == 2:
                # Remove the newline at the end, make it uppercase, and save it
                sequences.append(line.strip().upper())

    return sequences

In [3]:
"""De Bruijn Graph Genome Assembly Module.

This module provides classes and functions for constructing De Bruijn graphs
from sequencing reads and performing genome assembly using Eulerian path
traversal.
"""
from collections import defaultdict
import random
import sys

class DeBruijnGraph:
    """Main class for De Bruijn graphs and genome assembly.

    This class builds De Bruijn graphs from sequencing reads and performs
    genome assembly by finding Eulerian paths through connected components
    of the graph.

    Attributes:
        graph (defaultdict): Adjacency list representation of graph edges.
            Keys are (k-1)-mers, values are lists of adjacent (k-1)-mers.
        k (int): The k-mer size used for graph construction.

    Example:
        >>> reads = ["ATGGCGTACG", "GCGTACGTTA", "ACGTTACCAT"]
        >>> dbg = DeBruijnGraph(reads, k=6)
        >>> contigs = dbg.assemble_contigs(seed=42)
        >>> len(contigs) > 0
        True
    """

    # Set up: makes blank map and saves k
    def __init__(self, reads, k):
        """Initialize De Bruijn graph from sequencing reads.

        Args:
            reads (list): List of DNA sequence strings.
            k (int): K-mer size for graph construction.

        Example:
            >>> reads = ["ATGGCG", "GCGTGC", "TGCAAC"]
            >>> dbg = DeBruijnGraph(reads, k=4)
            >>> len(dbg.graph) > 0
            True
        """
        self.graph = defaultdict(list)
        self.k = k
        self.build_graph_from_reads(reads, k)

    # Helper function: Each k-mer becomes an edge from its left (k−1)-mer to its right (k−1)-mer
    def add_edge(self, left, right):
        """Add a directed edge to the graph.

        Args:
            left (str): Source (k-1)-mer node.
            right (str): Destination (k-1)-mer node.

        Example:
            >>> dbg = DeBruijnGraph([], k=4)
            >>> dbg.add_edge("ATG", "TGG")
            >>> "TGG" in dbg.graph["ATG"]
            True
        """
        # Find the list of dots that "left" points to (an empty list is made if "left" is new),
        # then add "right" to the end of that list
        self.graph[left].append(right)

    def remove_edge(self, left, right):
        """Remove a directed edge from the graph.

        Args:
            left (str): Source (k-1)-mer node.
            right (str): Destination (k-1)-mer node.

        Example:
            >>> dbg = DeBruijnGraph([], k=4)
            >>> dbg.add_edge("ATG", "TGG")
            >>> dbg.remove_edge("ATG", "TGG")
            >>> len(dbg.graph["ATG"])
            0
        """
        # Get the list of dots that "left" points to; use an empty list if "left" isn't on the map
        # (.get() is used so we don't accidentally add "left" as a new dot)
        neighbors = self.graph.get(left, [])
        # Only try to erase the arrow if it actually exists
        if right in neighbors:
            # Erase one copy of "right" (the first one found); this also changes self.graph, because "neighbors" is the same list, not a copy
            neighbors.remove(right)

# ---------- Build the de Bruijn graph ----------
    def build_graph_from_reads(self, reads, k):
        """Build De Bruijn graph from multiple sequencing reads.

        Extracts all k-mers from all reads and adds edges between
        consecutive (k-1)-mers within each k-mer.

        Args:
            reads (list): List of DNA sequence strings.
            k (int): K-mer length for graph construction.

        Example:
            >>> reads = ["ATGGC", "TGGCA"]
            >>> dbg = DeBruijnGraph([], k=4)
            >>> dbg.build_graph_from_reads(reads, 4)
            >>> "ATG" in dbg.graph
            True
        """
        # Go through every read, one at a time
        for read in reads:
            # Slide a window across the read; the last start spot is len(read) - k
            for i in range(len(read) - k + 1):
                # Cut out the k-mer: k letters starting at position i
                kmer = read[i:i + k]
                # Left dot = every letter except the last one
                left = kmer[:-1]
                # Right dot = every letter except the first one
                right = kmer[1:]
                # Only add the edge if it is not already there (skip duplicates)
                if right not in self.graph[left]:
                    self.add_edge(left, right)

# ---------- Walk the graph ----------
    def eulerian_walk(self, node, graph, seed=None, backward_tour=None):
        """Perform recursive Eulerian walk on a graph component.

        This is a recursive function that follows all edges from a node
        to traverse the graph, building a path in reverse order.

        Args:
            node (str): Current node to traverse from.
            graph (defaultdict): Graph or subgraph to traverse.
            seed (int, optional): Seed for random edge selection.

        Returns:
            list: backward_tour: List of (k-1)-mers traversed (in reverse order).

        Example:
            >>> reads = ["ATGGCG"]
            >>> dbg = DeBruijnGraph(reads, k=4)
            >>> graph_copy = defaultdict(list, dbg.graph)
            >>> tour = dbg.eulerian_walk("ATG", graph_copy, seed=42)
            >>> len(tour) > 0
            True
        """
        # If a seed was given, set it so the "random" choices are the same every run
        if seed is not None:
            random.seed(seed)

        if backward_tour is None:
            backward_tour = []
        # Check once: does this node still have an edge going out?
        if graph.get(node):
            # Pick one item from this node's list (the next node to move to)
            next_node = random.choice(graph[node])
            # Cross that item off the list
            graph[node].remove(next_node)

            self.eulerian_walk(next_node, graph, backward_tour=backward_tour)
            # Dead end, write this dot down, go back, and append what was collected to list (come out backwards)
        backward_tour.append(node)
        return backward_tour

# ---------- Turn the walk into DNA ----------
    def tour_to_sequence(self, tour):
        """Convert a tour of (k-1)-mers into a DNA sequence.

        Args:
            tour (list): List of (k-1)-mer strings in order.

        Returns:
            str: Assembled DNA sequence.

        Example:
            >>> dbg = DeBruijnGraph([], k=4)
            >>> tour = ['ATG', 'TGG', 'GGC', 'GCG']
            >>> dbg.tour_to_sequence(tour)
            'ATGGCG'
        """
        # If tour is empty, there is nothing to spell out, so return an empty string
        if not tour:
            return ""
        # Start sequence with the first node, whole
        sequence = tour[0]
        # Go through every node after the first one
        for node in tour[1:]:
            # Add only the last letter of this node (the rest overlaps the dot before)
            sequence += node[-1]
        # Return sequence, the finished DNA string
        return sequence

# ---------- Assemble contigs ----------
    def assemble_contigs(self, seed=None, copy_graph=False):
        """Assemble all contigs from the De Bruijn graph.

        Finds all connected components and generates an Eulerian path
        for each component, producing multiple assembled contigs.

        Args:
            seed (int, optional): Random seed for reproducible assembly.
            copy_graph (bool, optional): If True, walk a copy so self.graph is kept. If False (default), walk self.graph directly to save memory, which empties it.
        Returns:
            list: List of assembled contig sequences (DNA strings).

        Example:
            >>> reads = ["ATGGCGTACG", "GCGTACGTTA", "ACGTTACCAT"]
            >>> dbg = DeBruijnGraph(reads, k=6)
            >>> contigs = dbg.assemble_contigs(seed=42)
            >>> all(isinstance(c, str) for c in contigs)
            True
        """
        # If a seed was given, set it once so the walks make the same choices every run
        if seed is not None:
            random.seed(seed)
        # Walk a copy if asked, so self.graph survives
        if copy_graph:
            graph = defaultdict(list, {node: list(edges) for node, edges in self.graph.items()})
        # Otherwise walk self.graph directly to save memory (this erases its edges)
        else:
            graph = self.graph
        # Tally sheet for how many edges come INTO each node (starts at 0 for every node)
        in_count = defaultdict(int)
        # Go through every node and the list of nodes it points to
        for node, edges in graph.items():
            # Go through each node it points to
            for next_node in edges:
                # Add 1 to that node's incoming count
                in_count[next_node] += 1

        # Most recursion the walk can need is one call per edge
        num_edges = sum(in_count.values())
        # Remember the current limit so we can put it back afterward
        old_limit = sys.getrecursionlimit()
        # Raise the limit just enough for this graph
        sys.setrecursionlimit(max(old_limit, num_edges + 1000))

        # Use try/finally so the limit is put back even if something crashes
        try:
            # Start nodes: edges going out > in (where a path begins)
            start_nodes = [node for node in graph if len(graph[node]) > in_count[node]]
            # After those, try every node, to catch loops that have no clear start
            start_nodes += list(graph)
            # Empty list to collect the finished contigs
            contigs = []
            # Try each start node, in order
            for start in start_nodes:
                # Keep walking from this node while it still has an unused edge going out
                while graph.get(start):
                    # Walk from this node (this erases the edges it uses)
                    backward_tour = self.eulerian_walk(start, graph)
                    # Flip the walk into forward order
                    tour = backward_tour[::-1]
                    # Spell the walk out as DNA and save it as one contig
                    contigs.append(self.tour_to_sequence(tour))
        # Put the recursion limit back to what it was
        finally:
            sys.setrecursionlimit(old_limit)
        # Return contigs, the list of all DNA strings found
        return contigs

# ---------- Remove reverse-complement duplicates ----------
    def remove_reverse_complement_duplicates(self, contigs, cutoff=0.9):
        """Remove contigs that are reverse-complement copies of other contigs.

        Reads come from both DNA strands, so most regions get assembled
        twice: once as a contig and once as its reverse complement. This
        goes through the contigs from longest to shortest and drops any
        contig whose k-mers mostly (at least cutoff) appear, as reverse
        complements, in contigs already kept.

        Args:
            contigs (list): List of contig sequences.
            cutoff (float): Fraction of k-mers that must already be seen
                (as reverse complements) for a contig to be dropped.

        Returns:
            list: kept, the contigs with reverse-complement copies removed
                (longest first).

        Example:
            >>> dbg = DeBruijnGraph([], k=4)
            >>> dbg.remove_reverse_complement_duplicates(["AAACGT", "ACGTTT"])
            ['AAACGT']
            >>> dbg.remove_reverse_complement_duplicates(["AAACGT", "GGGCCA"])
            ['AAACGT', 'GGGCCA']
        """
        # Table for swapping each letter with its partner on the other strand
        complement = str.maketrans("ACGT", "TGCA")
        # Every k-mer from the contigs we have kept so far
        seen = set()
        # The contigs we decide to keep
        kept = []
        # Go through the contigs from longest to shortest, so the longest copy is kept
        for contig in sorted(contigs, key=len, reverse=True):
            # Number of k-mers in this contig
            num_kmers = len(contig) - self.k + 1
            # Reverse complement of the whole contig (swap letters, then flip)
            rc_contig = contig.translate(complement)[::-1]
            # Count how many of its reverse-complement k-mers were already kept
            found = sum(rc_contig[i:i + self.k] in seen for i in range(num_kmers))
            # If most of them were, this contig is the other strand's copy, so skip it
            if num_kmers > 0 and found / num_kmers >= cutoff:
                continue
            # Otherwise keep this contig
            kept.append(contig)
            # Remember all of its k-mers for checking later contigs
            seen.update(contig[i:i + self.k] for i in range(num_kmers))
        # Return kept, the contigs without reverse-complement copies
        return kept

# ---------- Assembly statistics ----------
    def get_assembly_stats(self, contigs):
        """Calculate assembly statistics for assembled contigs.
        N50 is found by sorting contig lengths from longest to shortest and
        adding them up; N50 is the length of the contig that brings the
        running total to at least half of the total length
        Args:
            contigs (list): List of contig sequences.

        Returns:
            dict: Dictionary containing assembly statistics:
                - num_contigs: Total number of contigs
                - total_length: Total assembled sequence length
                - longest_contig: Length of longest contig
                - shortest_contig: Length of shortest contig
                - mean_length: Mean contig length
                - n50: N50 statistic

        Example:
            >>> contigs = ["ATGGCG", "TTTAAA", "CCCCCCCCCC"]
            >>> dbg = DeBruijnGraph([], k=4)
            >>> stats = dbg.get_assembly_stats(contigs)
            >>> stats['num_contigs']
            3
        """
         # List of contig lengths, sorted from longest to shortest
        lengths = sorted((len(contig) for contig in contigs), reverse=True)
        # If there are no contigs, return all zeros
        if not lengths:
            return {"num_contigs": 0, "total_length": 0, "longest_contig": 0,
                    "shortest_contig": 0, "mean_length": 0, "n50": 0}
        # Add up all the lengths
        total_length = sum(lengths)
        # Start the running total at 0
        running_total = 0
        # Start n50 at 0 until we find it
        n50 = 0
        # Go through the lengths, longest first
        for length in lengths:
            # Add this contig's length to the running total
            running_total += length
            # Once the running total reaches half of the total length...
            if running_total >= total_length / 2:
                # ...this contig's length is the N50
                n50 = length
                # Stop looking
                break
        # Put all the numbers into one dictionary called stats
        stats = {
            # How many contigs there are
            "num_contigs": len(lengths),
            # All the contig lengths added up
            "total_length": total_length,
            # The first length is the longest (the list is sorted longest first)
            "longest_contig": lengths[0],
            # The last length is the shortest
            "shortest_contig": lengths[-1],
            # The average length
            "mean_length": total_length / len(lengths),
            # The N50 found above
            "n50": n50,
        }
        # Return stats, the dictionary of numbers
        return stats

# ---------- Save the contigs ----------
    def write_fasta(self, contigs, filename):
        """Write assembled contigs to a FASTA file.

        Args:
            contigs (list): List of contig sequences.
            filename (str): Output FASTA filename.

        Example:
            >>> contigs = ["ATGGCG", "TTTAAA"]
            >>> dbg = DeBruijnGraph([], k=4)
            >>> dbg.write_fasta(contigs, "output.fasta")
        """
        # Open the file for writing ("w" creates it, or erases it if it already exists)
        with open(filename, "w") as out:
            # Go through each contig, numbering them 1, 2, 3, ...
            for i, contig in enumerate(contigs, 1):
                # Write the header line, like ">contig_1 length=6"
                out.write(f">contig_{i} length={len(contig)}\n")
                # Write the contig's DNA on the next line
                out.write(contig + "\n")
        pass


In [4]:
print("="*60)
print("EXAMPLE 1: Assembling 9 overlapping reads into one contig")
print("="*60)

toy_reads_1 = [
    "ATGGCGTACG",  # Read 1
    "GGCGTACGTT",  # Read 2: overlaps with Read 1
    "CGTACGTTAC",  # Read 3: overlaps with Read 2
    "TACGTTACCA",  # Read 4: overlaps with Read 3
    "CGTTACCATG",  # Read 5: overlaps with Read 4
    "TTACCATGGG",  # Read 6: overlaps with Read 5
    "ACCATGGGCC",  # Read 7: overlaps with Read 6
    "CATGGGCCTA",  # Read 8: overlaps with Read 7
    "TGGGCCTAAA"   # Read 9: overlaps with Read 8
]

# Build the graph from the 9 toy reads with k = 6 (runs Step 2)
dbg = DeBruijnGraph(toy_reads_1, k=6)
# Count all edges in the graph
num_edges = sum(len(edges) for edges in dbg.graph.values())
# Show how many nodes have edges going out, and how many edges there are
print("Nodes with outgoing edges:", len(dbg.graph))
print("Edges:", num_edges)

# Assemble the contigs (runs Steps 3, 4 and 5)
contigs = dbg.assemble_contigs(seed=42)
# Show the contigs
print("Contigs:", contigs)

# Check the answer against the sequence the toy reads came from
expected = "ATGGCGTACGTTACCATGGGCCTAAA"
print("Correct:", contigs == [expected])

# Calculate the statistics (runs Step 6)
stats = dbg.get_assembly_stats(contigs)
# Show the statistics
print("Stats:", stats)

# Save the contigs to a FASTA file (runs Step 7)
dbg.write_fasta(contigs, "toy_assembly.fasta")
# Read the file back and show what is in it
print(open("toy_assembly.fasta").read())

EXAMPLE 1: Assembling 9 overlapping reads into one contig
Nodes with outgoing edges: 21
Edges: 21
Contigs: ['ATGGCGTACGTTACCATGGGCCTAAA']
Correct: True
Stats: {'num_contigs': 1, 'total_length': 26, 'longest_contig': 26, 'shortest_contig': 26, 'mean_length': 26.0, 'n50': 26}
>contig_1 length=26
ATGGCGTACGTTACCATGGGCCTAAA



---
# Real data
This section utilizes a FASTQ file with 2.4 million "perfect" 150 bp reads simulated
from the _S. cerevisiae_ (yeast) genome (`R64`) and tasks your program to ingest, assemble, and traverse
the genome with statistical output report.

In [5]:
"""Driver program for genome assembly using De Bruijn graphs.

This cell runs the completed DeBruijnGraph class on a FASTQ file of perfect
(error-free, variant-free) single-end reads simulated from a reference genome,
then reports assembly statistics.

Usage:
    Run all cells above this one first, then set the configuration values
    below and run this cell.

Expected input:
    - FASTQ or gzipped FASTQ (.fq, .fastq, .fq.gz, .fastq.gz)
    - Perfect single-end reads simulated with wgsim

Output:
    - <output_fasta>: Assembled contigs
    - <stats_file>: Detailed statistics report
"""

import time
from datetime import datetime

# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------
INPUT_FILE = "data/sacCer_R64_150SE_30x_wgsim.fq.gz"
OUTPUT_FASTA = "genome_assembly.fasta"
STATS_FILE = "genome_assembly_stats.txt"
K_MER_SIZE = 31
# Size (bp) of the reference the reads were simulated from; used for the
# coverage and assembly-fraction estimates. Set to None if unknown.
GENOME_SIZE = 12_071_326


def _pct(part, whole):
    """Return part/whole as a percentage, or 0.0 when whole is zero."""
    return (part / whole * 100) if whole else 0.0


def write_statistics_file(
    stats_file,
    input_file,
    num_reads,
    avg_read_length,
    k_mer_size,
    num_nodes,
    num_edges,
    stats,
    contig_lengths,
    timing,
    genome_size,
    coverage_estimate,
    assembly_fraction,
):
    """Write comprehensive assembly statistics to a text file.

    Args:
        stats_file (str): Output filename.
        input_file (str): Input FASTQ filename.
        num_reads (int): Number of reads processed.
        avg_read_length (float): Average read length.
        k_mer_size (int): K-mer size used.
        num_nodes (int): Number of graph nodes.
        num_edges (int): Number of distinct graph edges.
        stats (dict): Assembly statistics.
        contig_lengths (list): Contig lengths sorted longest first.
        timing (dict): Timing information, including 'total_time'.
        genome_size (int or None): Reference genome size in bp.
        coverage_estimate (float or None): Estimated sequencing coverage.
        assembly_fraction (float or None): Assembly size as % of genome.
    """
    rule = "-" * 80 + "\n"
    banner = "=" * 80 + "\n"

    with open(stats_file, "w") as f:
        f.write(banner)
        f.write("GENOME ASSEMBLY STATISTICS\n")
        f.write(banner + "\n")

        f.write(f"Analysis date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
        f.write(f"Input file:    {input_file}\n")
        f.write(f"K-mer size:    {k_mer_size}\n\n")

        f.write(rule + "INPUT DATA\n" + rule)
        f.write(f"Number of reads:         {num_reads:,}\n")
        f.write(f"Average read length:     {avg_read_length:.1f} bp\n")
        f.write(f"Total sequencing data:   {num_reads * avg_read_length:,.0f} bp\n")
        if genome_size:
            f.write(f"Reference genome size:   {genome_size:,} bp\n")
            f.write(f"Estimated coverage:      {coverage_estimate:.1f}x\n")
        f.write(f"Read time:               {timing['read_time']:.2f} seconds\n\n")

        f.write(rule + "DE BRUIJN GRAPH CONSTRUCTION\n" + rule)
        f.write(f"Graph nodes:             {num_nodes:,}\n")
        f.write(f"Graph edges:             {num_edges:,}\n")
        avg_degree = num_edges / num_nodes if num_nodes else 0.0
        f.write(f"Average out-degree:      {avg_degree:.2f}\n")
        f.write(f"Construction time:       {timing['graph_time']:.2f} seconds\n\n")

        f.write(rule + "ASSEMBLY RESULTS\n" + rule)
        f.write(f"Number of contigs:       {stats['num_contigs']:,}\n")
        f.write(f"Total assembly length:   {stats['total_length']:,} bp\n")
        if genome_size:
            f.write(f"Assembly vs. genome:     {assembly_fraction:.2f}%\n")
        f.write(f"Longest contig:          {stats['longest_contig']:,} bp\n")
        f.write(f"Shortest contig:         {stats['shortest_contig']:,} bp\n")
        f.write(f"Mean contig length:      {stats['mean_length']:,.1f} bp\n")
        f.write(f"N50:                     {stats['n50']:,} bp\n")
        f.write(f"Assembly time:           {timing['assembly_time']:.2f} seconds\n\n")

        f.write(rule + "TOP 20 LONGEST CONTIGS\n" + rule)
        for i, length in enumerate(contig_lengths[:20], 1):
            f.write(f"{i:3d}. {length:10,} bp\n")
        f.write("\n")

        f.write(rule + "CONTIG LENGTH DISTRIBUTION\n" + rule)
        bins = [
            ("all", len(contig_lengths)),
            (">100kb", sum(1 for x in contig_lengths if x > 100_000)),
            (">50kb", sum(1 for x in contig_lengths if x > 50_000)),
            (">10kb", sum(1 for x in contig_lengths if x > 10_000)),
            (">5kb", sum(1 for x in contig_lengths if x > 5_000)),
            (">1kb", sum(1 for x in contig_lengths if x > 1_000)),
            (">500bp", sum(1 for x in contig_lengths if x > 500)),
        ]
        for bin_name, count in bins:
            f.write(f"Contigs {bin_name:8s}:     {count:,}\n")
        f.write("\n")

        total_time = timing["total_time"]
        f.write(rule + "TIMING SUMMARY (excluding file output)\n" + rule)
        for label, key in (
            ("Read time:", "read_time"),
            ("Graph construction:", "graph_time"),
            ("Assembly:", "assembly_time"),
        ):
            f.write(f"{label:25s}{timing[key]:8.2f} seconds "
                    f"({_pct(timing[key], total_time):5.1f}%)\n")
        f.write(f"{'Total time:':25s}{total_time:8.2f} seconds "
                f"({total_time / 60:.2f} minutes)\n\n")

        f.write(banner + "END OF REPORT\n" + banner)


def count_graph(graph):
    """Count nodes and distinct edges in an adjacency-list graph.

    Nodes that only receive edges may not be keys of the adjacency dict, so
    they are counted separately. Duplicate neighbor entries (the same k-mer
    seen in several reads) are counted once.

    Args:
        graph (dict): Mapping of (k-1)-mer -> iterable of neighbor (k-1)-mers.

    Returns:
        tuple: (num_nodes, num_edges)
    """
    num_edges = 0
    sink_only = set()
    for neighbors in graph.values():
        distinct = set(neighbors)
        num_edges += len(distinct)
        for node in distinct:
            if node not in graph:  # membership test does not insert keys
                sink_only.add(node)
    return len(graph) + len(sink_only), num_edges


def assemble_genome(
    input_file=INPUT_FILE,
    output_fasta=OUTPUT_FASTA,
    stats_file=STATS_FILE,
    k_mer_size=K_MER_SIZE,
    genome_size=GENOME_SIZE,
):
    """Run the complete assembly pipeline.

    1. Load FASTQ reads
    2. Build the De Bruijn graph
    3. Assemble contigs
    4. Calculate statistics
    5. Write output files

    Returns:
        dict: Assembly results with keys 'dbg', 'contigs', 'stats', 'timing',
            'graph_stats', 'coverage', 'output_fasta', and 'stats_file'.
    """
    print("=" * 80)
    print("GENOME ASSEMBLY PIPELINE")
    print("=" * 80)
    print(f"Start time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")

    timing = {}

    # ------------------------------------------------------------------ 1
    print("STEP 1: Loading sequencing reads")
    print("-" * 80)
    print(f"Input file: {input_file}\n")

    start_time = time.time()
    reads = read_fastq(input_file)
    timing["read_time"] = time.time() - start_time

    num_reads = len(reads)
    if num_reads == 0:
        raise ValueError(f"No reads found in {input_file}")
    total_bases = sum(len(read) for read in reads)
    avg_read_length = total_bases / num_reads

    print(f"Reads loaded: {num_reads:,}")
    print(f"Total bases: {total_bases:,} bp")
    print(f"Average read length: {avg_read_length:.1f} bp")
    print(f"Time elapsed: {timing['read_time']:.2f} seconds")
    print("\nSample reads:")
    print(f"  First: {reads[0][:80]}...")
    print(f"  Last:  {reads[-1][:80]}...\n")

    # ------------------------------------------------------------------ 2
    print("STEP 2: Building De Bruijn graph")
    print("-" * 80)
    print(f"K-mer size: {k_mer_size}")
    print(f"Building graph from {num_reads:,} reads...\n")

    start_time = time.time()
    dbg = DeBruijnGraph(reads, k=k_mer_size)
    timing["graph_time"] = time.time() - start_time

    del reads  # the graph holds everything needed; free the read list

    num_nodes, num_edges = count_graph(dbg.graph)
    avg_degree = num_edges / num_nodes if num_nodes else 0.0

    print("Graph construction complete!")
    print(f"  Nodes (distinct {k_mer_size - 1}-mers): {num_nodes:,}")
    print(f"  Edges (distinct {k_mer_size}-mers):   {num_edges:,}")
    print(f"  Average out-degree: {avg_degree:.2f}")
    print(f"Time elapsed: {timing['graph_time']:.2f} seconds\n")

    # ------------------------------------------------------------------ 3
    print("STEP 3: Assembling contigs")
    print("-" * 80)
    print("Traversing the graph to build contigs...\n")

    start_time = time.time()
    contigs = dbg.assemble_contigs()
    contigs = dbg.remove_reverse_complement_duplicates(contigs)
    timing["assembly_time"] = time.time() - start_time
    timing["total_time"] = (
        timing["read_time"] + timing["graph_time"] + timing["assembly_time"]
    )

    print("Assembly complete!")
    print(f"  Contigs generated: {len(contigs):,}")
    print(f"Time elapsed: {timing['assembly_time']:.2f} seconds\n")

    # ------------------------------------------------------------------ 4
    print("STEP 4: Calculating assembly statistics")
    print("-" * 80)

    stats = dbg.get_assembly_stats(contigs)
    contig_lengths = sorted((len(c) for c in contigs), reverse=True)

    print("Assembly statistics:")
    print(f"  Number of contigs:     {stats['num_contigs']:,}")
    print(f"  Total assembly length: {stats['total_length']:,} bp")
    print(f"  Longest contig:        {stats['longest_contig']:,} bp")
    print(f"  Shortest contig:       {stats['shortest_contig']:,} bp")
    print(f"  Mean contig length:    {stats['mean_length']:,.1f} bp")
    print(f"  N50:                   {stats['n50']:,} bp\n")

    print("Top 10 longest contigs:")
    for i, length in enumerate(contig_lengths[:10], 1):
        print(f"  {i:2d}. {length:,} bp")
    print()

    if genome_size:
        coverage_estimate = total_bases / genome_size
        assembly_fraction = _pct(stats["total_length"], genome_size)
        print("Genome coverage analysis:")
        print(f"  Reference genome size:         {genome_size:,} bp")
        print(f"  Estimated sequencing coverage: {coverage_estimate:.1f}x")
        print(f"  Assembly size vs. genome:      {assembly_fraction:.1f}%")
        print("  (Near 200% usually means reverse-complement duplicate "
              "contigs were not removed.)\n")
    else:
        coverage_estimate = assembly_fraction = None

    # ------------------------------------------------------------------ 5
    print("STEP 5: Writing output files")
    print("-" * 80)

    dbg.write_fasta(contigs, output_fasta)
    print(f"✓ Contigs written to: {output_fasta}")

    write_statistics_file(
        stats_file,
        input_file,
        num_reads,
        avg_read_length,
        k_mer_size,
        num_nodes,
        num_edges,
        stats,
        contig_lengths,
        timing,
        genome_size,
        coverage_estimate,
        assembly_fraction,
    )
    print(f"✓ Statistics written to: {stats_file}\n")

    print("=" * 80)
    print("ASSEMBLY COMPLETE")
    print("=" * 80)
    print(f"Total time (excluding file output): {timing['total_time']:.2f} seconds "
          f"({timing['total_time'] / 60:.2f} minutes)")
    print(f"End time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")

    return {
        "dbg": dbg,
        "contigs": contigs,
        "stats": stats,
        "timing": timing,
        "graph_stats": {
            "num_nodes": num_nodes,
            "num_edges": num_edges,
            "avg_degree": avg_degree,
        },
        "coverage": coverage_estimate,
        "output_fasta": output_fasta,
        "stats_file": stats_file,
    }


print("\n" + "=" * 80)
print("GENOME DE BRUIJN GRAPH ASSEMBLY")
print("Student Assignment Driver Program")
print("=" * 80 + "\n")

result = assemble_genome()

print("=" * 80)
print("FIVE LONGEST ASSEMBLED CONTIGS")
print("=" * 80 + "\n")

for i, contig in enumerate(sorted(result["contigs"], key=len, reverse=True)[:5], 1):
    preview = contig[:100] + "..." if len(contig) > 100 else contig
    print(f">contig_{i} length={len(contig)}")
    print(preview + "\n")

print("=" * 80)
print("✓ ASSEMBLY COMPLETE")
print("=" * 80)
print("\nResults saved to:")
print(f"  • {result['output_fasta']} - {result['stats']['num_contigs']:,} assembled contigs")
print(f"  • {result['stats_file']} - detailed assembly statistics")
print("\nKey metrics:")
print(f"  • Total assembly: {result['stats']['total_length']:,} bp")
print(f"  • N50: {result['stats']['n50']:,} bp")
print(f"  • Longest contig: {result['stats']['longest_contig']:,} bp")
if result["coverage"] is not None:
    print(f"  • Coverage: {result['coverage']:.1f}x")
print()


GENOME DE BRUIJN GRAPH ASSEMBLY
Student Assignment Driver Program

GENOME ASSEMBLY PIPELINE
Start time: 2026-10-13 19:38:56

STEP 1: Loading sequencing reads
--------------------------------------------------------------------------------
Input file: data/sacCer_R64_150SE_30x_wgsim.fq.gz

Reads loaded: 2,414,266
Total bases: 362,139,900 bp
Average read length: 150.0 bp
Time elapsed: 3.84 seconds

Sample reads:
  First: ATCACAAAGGAGACAAAATCGTTGATAAAAAGTGCATCAACATTCCCAGCCAGAAAATGCACATCATAAAGACATGTTAT...
  Last:  ATTTAGCTCGCTTGTTTCTTGATTTTTTAGTCTCAATTTCATATCGTTAATAAAATCCTTGTGAGCTTCTGAGTTTAAAG...

STEP 2: Building De Bruijn graph
--------------------------------------------------------------------------------
K-mer size: 31
Building graph from 2,414,266 reads...

Graph construction complete!
  Nodes (distinct 30-mers): 22,950,037
  Edges (distinct 31-mers):   22,957,882
  Average out-degree: 1.00
Time elapsed: 313.09 seconds

STEP 3: Assembling contigs
------------------------------------

In [6]:
# Tool that reports how much of the computer this Python process has used
import resource
# Peak memory used by this kernel so far (on a Mac this number is in bytes)
peak_bytes = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
# Convert bytes to gigabytes and print it
print(f"Peak memory: {peak_bytes / 1024**3:.2f} GB")

Peak memory: 4.74 GB
